In [1]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()
import requests


In [2]:
import sys

import logging
# 1) 全局基础配置（只需执行一次）
logging.basicConfig(
    level=logging.INFO,  # 全局最低级别：DEBUG/INFO/WARNING/ERROR/CRITICAL
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    stream=sys.stdout,   # 输出到 Notebook 的输出区
    force=True           # 覆盖已有配置（Notebook重跑时很有用）
)

# 2) 获取你模块的 logger（与代码一致）
log = logging.getLogger(__name__)
log.setLevel(logging.INFO)  # 可按需调高/调低

# 3) 现在日志会显示在 Notebook 输出区
log.info("Logging is configured. You should see this message.")


2026-03-10 05:07:29,561 - __main__ - INFO - Logging is configured. You should see this message.


## Hana Database

### Create the connection to Hana database

In [11]:
from hdbcli import dbapi
# Initial the cursor
connection=dbapi.connect(
        address="e41c3eb7-55e9-47db-8915-e1ab64b9872a.hna0.prod-eu10.hanacloud.ondemand.com",
        port="443",
        user="USR_7K4HHI6O79L4LB691X7CN6MUQ",
        password="Jx8Q6g6l7NSZIMnooxdlCdkCLdEiR9--w4NSU4A0uZKfvLbrjKkgCJ9V2BXRHkQKP9ENitstBv4nv8.OraOa27HlVGwl50.D9BoaNPhoUDl-oGEbek1.n7QmYm-i3r.d",
        autocommit=True,
        sslValidateCertificate=False
    )

In [7]:
# (Optional)Check the connetion
connection.isconnected()

True

In [14]:
# Initial the cursor
cursor = connection.cursor() 

### HANA database table

#### Create a table

In [36]:
# Create a custom table with attribute

table_name = "LANGCHAIN_DEMO_SELF_QUERY"
try:
  cursor.execute(
      f'''
      CREATE TABLE "{table_name}" (
        "id"        INTEGER PRIMARY KEY,
        "name"      NVARCHAR(100),
        "is_active" BOOLEAN,
        "height"    DOUBLE,
        "VEC_TEXT"  NCLOB,
        "VEC_META"  NCLOB,
        "VEC_VECTOR" REAL_VECTOR(768)
      )
      '''
  )
  print(f'Table "{table_name}" created in the SAP HANA database.')

except Exception:
    print(f"Table  {table_name} already exsits.")
    pass

Table "LANGCHAIN_DEMO_SELF_QUERY" created in the SAP HANA database.


![](./images/TableCreation.png)

#### Delete a table

In [37]:
# Delete exsiting table if exists
try:
    cursor.execute(f"DROP TABLE {table_name}")
    print("Table dropped successfully.")
except Exception:
    print("No existing table.")
    pass

Table dropped successfully.


In [39]:
# Delete exsiting table if exists
try:
    cursor.execute(f"DROP TABLE {table_name}")
    print("Table dropped successfully.")
except Exception:
    print("No existing table.")
    pass

No existing table.


## Simple RAG Case: Local Document

### Prepare Document

In [8]:
# Step 1: Load documents

from langchain_community.document_loaders import PyPDFDirectoryLoader
DATA_PATH = r"datafiles"
loader = PyPDFDirectoryLoader(DATA_PATH)
documents = loader.load()
print(f"Loaded {len(documents)} documents.")

Loaded 6 documents.


In [9]:
# Step 2: Chunk documents

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    length_function=len,
)
split_documents = text_splitter.split_documents(documents)
print(f"Split into {len(split_documents)} chunks.")

Split into 25 chunks.


### Embedding

In [13]:
# Step 3: Set embedding model  
 
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI, OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    deployment_id=variables.EMBEDDING_DEPLOYMENT_ID
    )  # Deployment ID of text-embedding-3-large

In [14]:
from langchain_hana import HanaDB

# Step 4: Define the embedding table 
table_name="TEST_EMBEDDING_TABLE"
db = HanaDB(
    embedding=embedding_model, 
    connection=connection, 
    table_name=table_name
)

# Step 5: Add embeded chunks into the table.  
db.add_documents(split_documents)
print(f"Table {db.table_name} created in the SAP HANA database.")

2026-03-10 05:11:33,293 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
2026-03-10 05:11:33,709 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
Table TEST_EMBEDDING_TABLE created in the SAP HANA database.


<i><b>db.add</b></i> will create the table using the given table name if it does not exsit.
The table is defaulted with 3 fields:
* A column VEC_TEXT, which contains the text of the Document.
* A column VEC_META, which contains the metadata of the Document.
* A column VEC_VECTOR, which contains the embeddings-vector of the Document’s text.

![](./images/db_add.png)</p>
If the table already exsits, <i><b>db.add</b></i> will add new entries into the table.
![](./images/DuplicateEntries.png)</p>

In [ ]:
# (Optional) delete this table for repeating test
try:
    cursor.execute(f'DROP TABLE "{table_name}"')
    print(f'Table "{table_name}" dropped successfully.')
except Exception:
    print(f'Table "{table_name}" does not exist.')


Table "TEST_EMBEDDING_TABLE" dropped successfully.


### Check the embeddings in SAP HANA Cloud Vector Engine 

In [ ]:
from IPython.display import Markdown
 
# Use `db.table_name` instead of `variables.EMBEDDING_TABLE` because HANA driver sanitizes a table name by removing unaccepted characters
is_ok = cursor.execute(
    f'''
    SELECT "VEC_TEXT", "VEC_META", TO_NVARCHAR("VEC_VECTOR") FROM "{table_name}"
    ''')

record_columns=cursor.fetchone()

if record_columns:
    display({"VEC_TEXT" : record_columns[0], "VEC_META" : eval(record_columns[1]), "VEC_VECTOR" : record_columns[2]})


{'VEC_TEXT': 'Introduction \nWe SAP are excited to announce that we have started working on a VS Code extension for \nABAP . We understand that the community has high expectations, and we want to \ncommunicate transparently about what you can expect from ABAP Development Tools for \nVS Code. In this article, we will share details about the scope of the ﬁrst release and what’s \nplanned for the future. \nSee related article: Behind the Design: How We Transformed the ABAP Development Tools',
 'VEC_META': {'producer': 'Microsoft: Print To PDF',
  'creator': 'PyPDF',
  'creationdate': '2025-11-18T10:40:21+08:00',
  'author': 'SUN Yufeng (BD/PTD-SPR1)',
  'moddate': '2025-11-18T10:40:21+08:00',
  'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx',
  'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf',
  'total_pages': 2,
  'page': 0,
  'page_label': '1'},
 'VEC_VECTOR': '[0.011229125,-0.02020525,-0.009772568,

### Run RAG

In [68]:
from gen_ai_hub.proxy.langchain.init_models import init_llm
model = init_llm(
    'gpt-4o', 
    temperature=0.1,
    max_tokens=8000
)


In [ ]:
# Alternatively
model = ChatOpenAI(
    deployment_id=variables.LLM_DEPLOYMENT_ID
)  # LLM deployment ID. Here gpt-4o has been maintained

In [69]:
# Create a retriever instance of the vector store
retriever = db.as_retriever(
    search_kwargs={"k": 1}
)

In [ ]:

from langchain.chains import RetrievalQA
# Create the QA instance to query llm based on custom documents
qa = RetrievalQA.from_llm(
    llm=model, 
    retriever=retriever, 
    return_source_documents=True)

# Send query
query = "Why is ABAP in VS Code is so appealing?"

answer = qa.invoke(query)
display(answer["result"])


'ABAP in Visual Studio Code is appealing because it provides more flexibility in choosing a development environment, addressing the demand from users for support beyond SAP GUI and Eclipse. Visual Studio Code is a popular IDE known for its versatility, ease of use, and extensive range of extensions, which can enhance the development experience compared to existing tools. Additionally, it aligns with the preferences of a significant number of users based on survey results from 2023 and 2025.'

In [ ]:

# Check addition information from retrievr 
for document in answer['source_documents']:
    display(document.metadata)   
    print(document.page_content)



{'producer': 'Microsoft: Print To PDF',
 'creator': 'PyPDF',
 'creationdate': '2025-11-18T10:42:07+08:00',
 'author': 'SUN Yufeng (BD/PTD-SPR1)',
 'moddate': '2025-11-18T10:42:07+08:00',
 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx',
 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf',
 'total_pages': 4,
 'page': 0,
 'page_label': '1'}

Introduction 
Currently, oƯicial ABAP tool support exists for SAP GUI and Eclipse. For years, users 
have asked us to bring this support to additional IDEs. Based on the user survey 
results from 2023 and 2025, the most requested development environment is Visual 
Studio Code (VS Code). However, many users have also expressed interest in other 
environments such as JetBrains IDEs, Neovim, or even Zed. In short, our user base 
wants more ﬂexibility when choosing their development environment.


## RAG Case: Online Document

In [144]:
TABLE_NAME = "GIT_DOCS"
LLM_MODEL_NAME = 'gpt-4o'
EMBEDDINGS_MODEL_NAME ='text-embedding-3-large'
GIT_URL="https://github.com/SAP/terraform-provider-btp"

### Prepare Document

#### Load document from git repository

In [4]:
from langchain_community.document_loaders import GitLoader

# Define a function to fetch file from a git respository
def fetch_gitrepository_docs(gitrepository_url):
    try:
        log.info("Getting the documents from the GitHub repository: %s", gitrepository_url)
        loader = GitLoader(
            clone_url=gitrepository_url,
            repo_path="./gen/docs/",
            file_filter=lambda file_path: file_path.startswith("./gen/docs/docs")
            and file_path.endswith(".md"),
            branch="main",
        )
        documents = loader.load()
        log.info("Documents loaded successfully. count=%d", len(documents) if documents else 0)
        return documents
    except Exception as e:
        log.error(f"Error occurred while loading documents: {str(e)}")


In [28]:
# Test the function
fetch_gitrepository_docs(gitrepository_url=GIT_URL)

2026-03-10 03:13:37,290 - __main__ - INFO - Getting the documents from the GitHub repository: https://github.com/SAP/terraform-provider-btp
2026-03-10 03:13:39,585 - __main__ - INFO - Documents loaded successfully. count=128


[Document(metadata={'source': 'docs/index.md', 'file_path': 'docs/index.md', 'file_name': 'index.md', 'file_type': '.md'}, page_content='---\npage_title: "SAP BTP Provider"\nsubcategory: ""\ndescription: |-\n  The Terraform provider for SAP BTP enables you to automate the provisioning, management, and configuration of resources on SAP Business Technology Platform https://account.hana.ondemand.com/. By leveraging this provider, you can simplify and streamline the deployment and maintenance of BTP services and applications.\n---\n# Terraform Provider for SAP BTP\n\nThe Terraform provider for SAP BTP enables you to automate the provisioning, management, and configuration of resources on [SAP Business Technology Platform](https://account.hana.ondemand.com/). By leveraging this provider, you can simplify and streamline the deployment and maintenance of BTP services and applications.\n\n## Example Usage\n\n```terraform\nterraform {\n  required_providers {\n    btp = {\n      source  = "SAP/b

#### Chunk the document 

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

def split_docs_into_chunks(
    documents: list[Document], chunk_size: int = 1000, chunk_overlap: int = 100
):
    """
    Splits a list of documents into chunks of specified size with overlap.

    Args:
        documents (list[Document]): The list of documents to be split into chunks.
        chunk_size (int, optional): The size of each chunk. Defaults to 1000.
        chunk_overlap (int, optional): The overlap between consecutive chunks. Defaults to 100.

    Returns:
        list[list[Document]]: A list of chunks, where each chunk is a list of documents.

    """
    try:
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            add_start_index=True,
        )
        chunks = text_splitter.split_documents(documents)
        log.info(f"Split {len(documents)} documents into {len(chunks)} chunks.")

        return chunks
    except Exception as e:
        log.error(f"An error occurred while splitting documents into chunks: {str(e)}")
        raise

In [32]:
# Test the function
git_docs=fetch_gitrepository_docs(gitrepository_url=GIT_URL)
chunks=split_docs_into_chunks(git_docs)

2026-03-10 03:15:21,069 - __main__ - INFO - Getting the documents from the GitHub repository: https://github.com/SAP/terraform-provider-btp
2026-03-10 03:15:23,359 - __main__ - INFO - Documents loaded successfully. count=128
2026-03-10 03:15:23,366 - __main__ - INFO - Split 128 documents into 516 chunks.


### Prepare language model

In [122]:
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from gen_ai_hub.proxy.langchain.openai import OpenAIEmbeddings
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
from langchain_core.rate_limiters import InMemoryRateLimiter

# Define a function to define the  chat llm and embedding model to be used
def create_llm_and_embeddings(llm_model,embedding_model):
    
    # Get the proxy client for the AI Core service
    proxy_client = get_proxy_client("gen-ai-hub")

    rate_limiter = InMemoryRateLimiter(
        requests_per_second=0.5,  # We can only make a request once every 5 seconds
        check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
        max_bucket_size=10,  # Controls the maximum burst size.
    )

    llm = ChatOpenAI(
        proxy_model_name=llm_model,
        proxy_client=proxy_client,
        temperature=0.1,
        rate_limiter=rate_limiter,
    )

    embeddings = OpenAIEmbeddings(
        proxy_model_name=embedding_model,
        proxy_client=proxy_client,
        show_progress_bar=True,
    )
    return llm, embeddings

In [51]:
# Test the function
llm,embeddings=create_llm_and_embeddings(
    llm_model=LLM_MODEL_NAME,
    embedding_model=EMBEDDINGS_MODEL_NAME,
)

print(llm)
print(embeddings)

rate_limiter=<langchain_core.rate_limiters.InMemoryRateLimiter object at 0x7fd9d9e4add0> client=<gen_ai_hub.proxy.native.openai.clients.ChatCompletions object at 0x7fd9cb6948d0> async_client=<gen_ai_hub.proxy.native.openai.clients.AsyncChatCompletions object at 0x7fd9cb695ed0> root_client=<gen_ai_hub.proxy.native.openai.clients.OpenAI object at 0x7fd9cb657ed0> root_async_client=<gen_ai_hub.proxy.native.openai.clients.AsyncOpenAI object at 0x7fd9d9eb87d0> model_name='gpt-4o' temperature=0.1 model_kwargs={} openai_api_key=SecretStr('**********') n=1 proxy_client=GenAIHubProxyClient(base_url=None, auth_url=None, client_id=None, client_secret=None, resource_group=None, ai_core_client=<ai_core_sdk.ai_core_v2_client.AICoreV2Client object at 0x7fd9db1ab0e0>) deployment_id='da74d08d2f277477' config_name='islm.dpl.cfg.ZTEST_GENAI_OPENAI.7AB55BB2EC021FD186C6F4447E709D45.Deployed based on model ZGPTOPENAI4 training 1' config_id='7eb56362-ff64-421e-9f91-cb765773ddf9' proxy_model_name='gpt-4o'
clie

### Embedding

#### Delete existing table content

In [123]:
# Define a function to check whether a table exists or not
def check_if_exists(table_name, schema_name="USR_7K4HHI6O79L4LB691X7CN6MUQ"):
    connection_to_hana = init_env.get_connection_to_hana_db()
    cursor = connection_to_hana.cursor()

    # Check if the table exists
    check_table_query ="""
    SELECT COUNT(*)
    FROM TABLES
    WHERE SCHEMA_NAME = ? AND TABLE_NAME = ?
    """

    cursor.execute(check_table_query, (schema_name, table_name))
    return cursor.fetchone()[0] > 0


In [98]:
# Test the function
check_if_exists(table_name=TABLE_NAME)

True

In [124]:
# Define a function to remove existing table
def teardown_hana_table(table_name):
    
    exists = check_if_exists(table_name=table_name)
    if exists is False:
        log.info(f"Table {table_name} does not exsit. Nothing to clean up.")
        return
    
    try:
        connection_to_hana = init_env.get_connection_to_hana_db()
        cursor = connection_to_hana.cursor()
        log.info(f"Dropping table {table_name}")
        cursor.execute(f"DROP TABLE {table_name}")
        cursor.close()
        log.info(f"Table {table_name} dropped successfully.")
    except Exception as e:
        log.error(type(e))
        log.error(f"Error dropping table: {str(e)}")


In [95]:
# Test the function
teardown_hana_table(table_name=TABLE_NAME)

2026-03-10 06:05:06,875 - __main__ - INFO - Table GIT_DOCS does not exsit. Nothing to clean up


#### Embedding and save to HANA table

In [125]:
from langchain_hana import HanaDB

# Define a function to do the embedding and then save the data into HANA table
def ingest(documents):
    try:
        teardown_hana_table(table_name=TABLE_NAME)
        log.info(f"Start ingesting data in {TABLE_NAME}")
        
        connection_to_hana = init_env.get_connection_to_hana_db()
        cursor = connection_to_hana.cursor()
        
        chunks = split_docs_into_chunks(documents=documents)

        _, embeddings = create_llm_and_embeddings(
            llm_model=LLM_MODEL_NAME,
            embedding_model=EMBEDDINGS_MODEL_NAME,
        )


        db = HanaDB(
            embedding=embeddings, 
            connection=connection_to_hana, 
            table_name=TABLE_NAME
        )
       
        log.info("Adding documents chunks to the HANA DB")
        db.add_documents(chunks)
        log.info("Documents added successfully.")
        log.info("Ingestion completed successfully.")

        cursor.close()
    except Exception as e:
        log.error(f"Error occurred during ingestion: {str(e)}")

In [126]:
# Run the ingest
git_docs=fetch_gitrepository_docs(gitrepository_url=GIT_URL)
ingest(documents=git_docs)


2026-03-10 07:04:36,259 - __main__ - INFO - Getting the documents from the GitHub repository: https://github.com/SAP/terraform-provider-btp
2026-03-10 07:04:39,326 - __main__ - INFO - Documents loaded successfully. count=128
2026-03-10 07:04:39,748 - __main__ - INFO - Dropping table GIT_DOCS
2026-03-10 07:04:39,763 - __main__ - INFO - Table GIT_DOCS dropped successfully.
2026-03-10 07:04:39,768 - __main__ - INFO - Start ingesting data in GIT_DOCS
2026-03-10 07:04:39,964 - __main__ - INFO - Split 128 documents into 516 chunks.
2026-03-10 07:04:40,313 - __main__ - INFO - Adding documents chunks to the HANA DB


  0%|          | 0/33 [00:00<?, ?it/s]

2026-03-10 07:04:40,870 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


  3%|▎         | 1/33 [00:00<00:16,  1.93it/s]

2026-03-10 07:04:41,501 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


...

2026-03-10 07:04:57,915 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


 97%|█████████▋| 32/33 [00:17<00:00,  1.87it/s]

2026-03-10 07:04:58,257 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 33/33 [00:17<00:00,  1.84it/s]


2026-03-10 07:04:58,483 - __main__ - INFO - Documents added successfully.
2026-03-10 07:04:58,484 - __main__ - INFO - Ingestion completed successfully.


### Run RAG

In [183]:
# Create a retriever

llm, embeddings = create_llm_and_embeddings(
    llm_model=LLM_MODEL_NAME,
    embedding_model=EMBEDDINGS_MODEL_NAME,
)

connection_to_hana = init_env.get_connection_to_hana_db()

db = HanaDB(
    embedding=embeddings, 
    connection=connection_to_hana, 
    table_name=TABLE_NAME
)
retriever = db.as_retriever(search_kwargs={"k":3 })

from langchain.chains import RetrievalQA

# Create the QA instance to query llm based on custom documents
qa = RetrievalQA.from_llm(
    llm=llm, 
    retriever=retriever, 
    return_source_documents=True)



In [185]:
# Send query
query = "What is btp_subaccount_destination_trust and how to use it?"
 

answer = qa.invoke(query)
print(answer["result"])








  0%|          | 0/1 [00:00<?, ?it/s]

2026-03-10 08:01:40,949 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9/embeddings?api-version=2025-03-01-preview "HTTP/1.1 200 OK"


100%|██████████| 1/1 [00:00<00:00,  2.56it/s]


2026-03-10 08:01:43,475 - httpx - INFO - HTTP Request: POST https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/da74d08d2f277477/chat/completions?api-version=2025-03-01-preview "HTTP/1.1 200 OK"
The `btp_subaccount_destination_trust` is a data source in the Terraform provider for SAP BTP that allows you to retrieve details about a specific subaccount destination trust. This data source is useful for obtaining information about the trust configuration between a subaccount and its destinations.

To use `btp_subaccount_destination_trust`, you need to have the appropriate permissions, such as Subaccount Administrator, Destination Administrator, Destination Viewer, or Connectivity and Destination Administrator.

Here is an example of how to use `btp_subaccount_destination_trust` in Terraform:

```terraform
# Read BTP Subaccount Destination Trust information for a specific subaccount and origin
data "btp_subaccount_destination_trust" "subaccount_dt_active" {
  